# 03 — First features

Computes pre-fight rolling features on top of the de-biased fight table from
notebook 02, closing leakage traps #2 and #3 from `ROADMAP.md`: the raw
per-fight stats (strikes, takedowns, control time...) are outcomes of the
fight they're attached to, so they can't be used directly. This notebook
rolls them up per fighter, using only fights strictly before the one being
predicted, and attaches the static (time-invariant) attributes — age,
height, reach, stance — on top.

Weight is deliberately excluded: it changes across a fighter's career as
they move divisions, so a single joined value wouldn't be safe the way
reach/stance are — `weight_class` on the fight itself is the correct source
if weight is ever needed.

**Day 3 changes.**

*All remaining raw stats rolled* — including the strike-location breakdown
(head/body/leg, distance/clinch/ground) and `reversals`. Note these are exact
decompositions of `sig_strikes` (verified: it equals head+body+leg and
distance+clinch+ground on 100% of rows), so expect collinearity.

*Attempts + accuracy instead of landed + attempted* — since
`landed = attempts × accuracy`, that's a reparameterisation rather than a
loss, but it separates "how much they throw" from "how well it lands".
Accuracy is built in both directions: the fighter's own, and how often
opponents land what they throw *at* them.

*Volume normalised by fight length, both directions* — a raw count conflates
skill with fight length. Attempt counts, knockdowns, submissions and
reversals become per-minute rates; control time becomes a **share of the
bout**, since it already shares units with fight duration. This applies to
both what a fighter does (offence) and what happens to them (defence).
`total_fight_time` is rolled too, giving each fighter's average bout length
— useful on its own as a finisher/distance-goer signal, and the denominator
these rates need.

*Reach-to-height ratio ("ape index")* — `reach_cm − height_cm`. A distinct
scouting concept from either alone: two fighters can share identical reach
but very different frames depending on height.

*Stance matchup* — `same_stance`, a single symmetric boolean rather than a
diff (stance is categorical, and "is this an open matchup" is the same fact
regardless of which fighter is labelled A or B). A directional flag assuming
southpaw or orthodox has the advantage was deliberately not built — that's a
hypothesis for the model to test, not something to hard-code.

*Prior title and 5-round fight counts* — cumulative, the same pattern as
`wins`/`losses`. Title status comes from `weight_class`; 5-round status adds
any fight (title or not) that ran past 900 seconds, since a 3-round fight
cannot physically exceed that — this is a lower bound, since a 5-round fight
that ends by an early finish is indistinguishable from a 3-round fight
ending the same way.

**Draws and no-contests are history too.** Notebook 02 now keeps all 7,840
fights, so a fighter's rolled averages, experience count and layoff include
the bouts that had no winner. They are removed only when modelling rows are
chosen, by `chronological_split`. `wins`/`losses` count neither; a draw is
half a win in `win_rate` and resets the streak; a no-contest is excluded
from `win_rate` and leaves the streak alone.

**Shrinkage.** Every feature that is a total divided by an amount of evidence
— accuracy, per-minute rates, control share, win rate, average fight time — is
shrunk towards the league value, so a tiny sample can't produce an extreme
number: one knockdown in an 8-second fight is not "a knockdown every 8
seconds". The strengths (alphas) are learned from the training years only
(2014–2019) and saved to `data/processed/alphas.json`, because a model must
always be paired with the alphas its features were built with. See
`features/shrinkage.py`.

As with notebook 02, the actual transform lives in
`src/ufc_predictor/features/` (`rolling.py`, `static.py`) — this notebook
calls it and checks the result.

In [1]:
from ufc_predictor.config import VALIDATION_START
from ufc_predictor.data.io import load_processed, load_fighters, save_processed
from ufc_predictor.features.rolling import attach_prior_features, fit_alphas
from ufc_predictor.features.shrinkage import save_alphas
from ufc_predictor.features.static import attach_static_features

fights = load_processed("fights")
fighters = load_fighters()

# how strongly to shrink each ratio feature - learned from the training years
# only, and saved next to the table it builds
alphas = fit_alphas(fights, until=VALIDATION_START)
save_alphas(alphas, until=VALIDATION_START)

featured = attach_prior_features(fights, alphas)
featured = attach_static_features(featured, fighters)
featured.shape

(7840, 367)

## Shrinkage strengths

`shrunk = (total + α × league) / (evidence + α)`. α is the amount of imaginary
league-average evidence every fighter starts with, counted in the feature's own
unit: attempts for accuracy, minutes for rates and control share, fights for win
rate and average fight time. Each feature's α was chosen by testing which value
best predicts what fighters did in their next fight (for win rate: who won),
using 2014–2019 only.

In [2]:
from ufc_predictor.features.rolling import RATIO_FEATURES
from ufc_predictor.features.shrinkage import describe_alphas

table = describe_alphas(RATIO_FEATURES, alphas)
print(f"{len(table)} shrunk features; the career versions:")
table[~table["feature"].str.contains("last3")].sort_values(["unit", "alpha"]).reset_index(drop=True)

91 shrunk features; the career versions:


,feature,alpha,unit
0,takedowns_absorbed_accuracy_career_avg,31.620,attempts
1,leg_strikes_accuracy_career_avg,100.000,attempts
2,body_strikes_absorbed_accuracy_career_avg,125.900,attempts
3,leg_strikes_absorbed_accuracy_career_avg,125.900,attempts
4,takedowns_accuracy_career_avg,125.900,attempts
5,body_strikes_accuracy_career_avg,158.500,attempts
6,clinchs_accuracy_career_avg,158.500,attempts
7,clinchs_absorbed_accuracy_career_avg,199.500,attempts
8,grounds_absorbed_accuracy_career_avg,199.500,attempts
9,dist_strikes_accuracy_career_avg,398.100,attempts


## Sanity checks

In [3]:
# no fights lost or duplicated by the merge
assert len(featured) == len(fights)

# a genuine debut fighter has no rolled-up history to draw on
assert featured["fighter_a_prior_is_debut"].sum() > 0
debut_rows = featured[featured["fighter_a_prior_is_debut"]]
assert debut_rows["fighter_a_prior_sig_strike_atts_per_min_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_sig_strikes_accuracy_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_total_fight_time_career_avg"].isna().all()

# shrinkage: a veteran's ratio features are never NaN any more - a fighter who
# never attempted something gets the league value. (A few veterans in 1994-97
# are NaN: the early UFC didn't record every stat, so there was no league value
# to shrink towards. None of them are in the modelling era.)
from ufc_predictor.config import ERA_START
from ufc_predictor.features.rolling import RATIO_FEATURES
era_veterans = featured[(featured["event_date"] >= ERA_START) & ~featured["fighter_a_prior_is_debut"]]
ratio_cols = [f"fighter_a_prior_{spec.name}" for spec in RATIO_FEATURES]
assert era_veterans[ratio_cols].notna().all().all()

# undecided fights are present, carry no label, and still have features
assert featured["fighter_a_won"].isna().sum() == 143
assert featured.loc[featured["fighter_a_won"].isna(), "diff_prior_fight_count"].notna().any()

any_debut_rate = (
    featured["fighter_a_prior_is_debut"] | featured["fighter_b_prior_is_debut"]
).mean()
print(f"fights involving at least one debutant: {any_debut_rate:.1%}")

fights involving at least one debutant: 25.7%


In [4]:
# leakage spot-check: Jim Miller's 3rd UFC fight (2009-03-07) should show a
# prior record of 2-0, not his eventual career record — this is the exact
# check that verified the module against a hand-computed example.
#
# he lands in the fighter_a slot for some fights and fighter_b for others
# (that's the point of the de-biasing in notebook 02), so both sides need
# checking, not just fighter_a.
jim_miller_id = 2500
mask = (featured["fighter_a_id"] == jim_miller_id) | (featured["fighter_b_id"] == jim_miller_id)
his_fights = featured[mask].sort_values("event_date")
his_3rd_fight = his_fights.iloc[2]
side = "fighter_a" if his_3rd_fight["fighter_a_id"] == jim_miller_id else "fighter_b"

prior_wins = his_3rd_fight[f"{side}_prior_wins"]
prior_losses = his_3rd_fight[f"{side}_prior_losses"]
print("date:", his_3rd_fight["event_date"].date())
print("prior_wins:", prior_wins)
print("prior_losses:", prior_losses)
assert prior_wins == 2
assert prior_losses == 0

date: 2009-03-07
prior_wins: 2.0
prior_losses: 0.0


In [5]:
# static feature sanity checks: row count preserved, diffs are simple
# subtraction, missingness matches what we already knew about the raw
# fighter table (reach ~45%, stance ~19%) rather than something worse
assert len(featured) == len(fights)

diff_pairs = {
    "diff_static_age": ("fighter_a_age_during", "fighter_b_age_during"),
    "diff_static_height_cm": ("fighter_a_height_cm", "fighter_b_height_cm"),
    "diff_static_reach_cm": ("fighter_a_reach_cm", "fighter_b_reach_cm"),
}
for diff_col, (a_col, b_col) in diff_pairs.items():
    assert featured[diff_col].equals(featured[a_col] - featured[b_col])

print("reach missing:", f"{featured['fighter_a_reach_cm'].isna().mean():.1%}")
print("stance missing:", f"{featured['fighter_a_stance'].isna().mean():.1%}")
print("age missing:", f"{featured['fighter_a_age_during'].isna().mean():.1%}")

reach missing: 7.5%
stance missing: 0.5%
age missing: 1.2%


In [6]:
# accuracy sanity checks: ratios must sit in [0, 1], and the defensive
# direction must carry information the raw absorbed volume doesn't — that
# near-zero correlation is the whole reason it was added.
#
# note the bound applies to the per-fighter columns only: diff_* columns are
# differences of two accuracies, so they legitimately span [-1, 1].
per_fighter_acc = [
    c for c in featured.columns
    if "_accuracy_" in c and c.startswith(("fighter_a_", "fighter_b_"))
]
print(f"{len(per_fighter_acc)} per-fighter accuracy columns")

for col in per_fighter_acc:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

# control time absorbed is a share of the bout, so it cannot exceed 1.0.
# it did before the denominator was fixed to cover only the bouts that
# actually recorded control time — see features/shrinkage.py.
for col in [c for c in featured.columns
            if "_frac_" in c and c.startswith(("fighter_a_", "fighter_b_"))]:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

both = featured[[
    "fighter_a_prior_sig_strike_atts_absorbed_per_min_career_avg",
    "fighter_a_prior_sig_strikes_absorbed_accuracy_career_avg",
]].dropna()
corr = both.corr().iloc[0, 1]
print(f"corr(absorbed volume/min, opponent accuracy against fighter): {corr:.3f}")

print("\nopponent accuracy against fighter (how hittable they are):")
print(featured["fighter_a_prior_sig_strikes_absorbed_accuracy_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

print("\nshare of bout spent controlled by the opponent:")
print(featured["fighter_a_prior_ctrl_time_absorbed_frac_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

72 per-fighter accuracy columns
corr(absorbed volume/min, opponent accuracy against fighter): -0.057

opponent accuracy against fighter (how hittable they are):
mean    0.434
std     0.035
min     0.306
max     0.604

share of bout spent controlled by the opponent:
mean    0.212
std     0.061
min     0.057
max     0.456


In [7]:
# item 1: ape index. real spread, and now a feature rather than left implicit
# in reach_cm and height_cm separately.
ape = featured["fighter_a_ape_index_cm"].dropna()
print(f"ape index (reach - height): mean {ape.mean():.1f}cm, std {ape.std():.1f}, "
      f"range {ape.min():.1f} to {ape.max():.1f}")

# item 3: stance matchup. a symmetric fact, not diffed.
print(f"\nsame_stance dtype: {featured['same_stance'].dtype}")
print(featured["same_stance"].value_counts(dropna=False))

# item 4: title / five-round counts are cumulative and monotonic per fighter,
# same shape as wins/losses. sanity-check against a known multi-title veteran.
top = featured.loc[featured["fighter_a_prior_title_fights"].idxmax()]
print(f"\nmost prior UFC title fights (fighter_a slot): {top['fighter_a_prior_title_fights']:.0f} "
      f"— {top['fighter_a_name']} on {top['event_date'].date()}")

assert featured["fighter_a_prior_five_round_fights"].ge(
    featured["fighter_a_prior_title_fights"]
).all(), "every title fight is 5-round, so five_round_fights can't be less than title_fights"

ape index (reach - height): mean 4.6cm, std 4.9, range -12.7 to 22.9

same_stance dtype: boolean
same_stance
True     4751
False    3019
<NA>       70
Name: count, dtype: Int64

most prior UFC title fights (fighter_a slot): 16 — Randy Couture on 2009-08-29


## A look at the engineered features

In [8]:
diff_cols = [c for c in featured.columns if c.startswith("diff_prior_") or c.startswith("diff_static_")]
print(f"{len(diff_cols)} diff features")
featured[diff_cols].describe().T[["mean", "std", "min", "max"]]

102 diff features


,mean,std,min,max
diff_prior_fight_count,-0.013648,6.455788,-41.000000,37.000000
diff_prior_wins,-0.011224,4.010527,-24.000000,21.000000
diff_prior_losses,-0.006633,2.785930,-16.000000,16.000000
diff_prior_win_rate,0.001027,0.115199,-0.478746,0.398742
diff_prior_streak,0.013393,2.279534,-16.000000,12.000000
...,...,...,...,...
diff_prior_total_fight_time_last3_avg,-2.359868,82.103031,-382.619026,311.431966
diff_static_age,-0.005464,5.286604,-17.000000,25.000000
diff_static_height_cm,0.078563,6.539941,-33.020000,33.020000
diff_static_reach_cm,0.091548,8.320091,-35.560000,33.020000


## Save

In [9]:
save_processed(featured, "fights_featured")
featured.head()

,weight_class,fights_url,event_url,event_date,fighter_a_age_during,fighter_b_age_during,fighter_a_body_strike_atts,fighter_b_body_strike_atts,fighter_a_body_strikes,fighter_b_body_strikes,...,fighter_a_stance,fighter_a_ape_index_cm,fighter_b_reach_cm,fighter_b_stance,fighter_b_ape_index_cm,diff_static_age,diff_static_height_cm,diff_static_reach_cm,diff_static_ape_index_cm,same_stance
0,Open Weight Bout,http://www.ufcstats.com/fight-details/17ee4caf...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,NaN,0,0,0,0,...,NaN,NaN,NaN,Southpaw,NaN,NaN,-17.78,NaN,NaN,<NA>
1,Open Weight Bout,http://www.ufcstats.com/fight-details/5655639b...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,24.0,27.0,0,0,0,0,...,Southpaw,NaN,NaN,Southpaw,NaN,-3.0,-5.08,NaN,NaN,True
2,Open Weight Bout,http://www.ufcstats.com/fight-details/aa161c71...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,27.0,0,0,0,0,...,Southpaw,NaN,NaN,Southpaw,NaN,NaN,5.08,NaN,NaN,True
3,Open Weight Bout,http://www.ufcstats.com/fight-details/c53325be...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,24.0,0,2,0,1,...,NaN,NaN,NaN,Southpaw,NaN,NaN,2.54,NaN,NaN,<NA>
4,Open Weight Bout,http://www.ufcstats.com/fight-details/8ea72cf8...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,30.0,0,1,0,1,...,Orthodox,NaN,NaN,Orthodox,NaN,NaN,-10.16,NaN,NaN,True


## Visualise the dataset

The full table is far too wide to read directly. A curated view of the most
recent fights: who fought, the label, and the headline engineered features —
striking volume, accuracy in both directions, how much of the bout they
spend controlled, and typical bout length.

In [10]:
readable_cols = [
    "event_date", "weight_class",
    "fighter_a_name", "fighter_b_name", "fighter_a_won",
    "diff_prior_win_rate", "diff_prior_streak",
    "diff_prior_sig_strike_atts_per_min_career_avg",       # striking volume, per minute
    "diff_prior_sig_strikes_accuracy_career_avg",          # striking accuracy (offence)
    "diff_prior_sig_strikes_absorbed_accuracy_career_avg", # how hittable they are
    "diff_prior_ctrl_time_frac_career_avg",                # share of bout THEY controlled
    "diff_prior_title_fights",                             # title-fight experience
    "diff_static_age", "diff_static_reach_cm", "diff_static_ape_index_cm",
    "same_stance",
]

view = featured[readable_cols].sort_values("event_date", ascending=False)
view.head(20)

,event_date,weight_class,fighter_a_name,fighter_b_name,fighter_a_won,diff_prior_win_rate,diff_prior_streak,diff_prior_sig_strike_atts_per_min_career_avg,diff_prior_sig_strikes_accuracy_career_avg,diff_prior_sig_strikes_absorbed_accuracy_career_avg,diff_prior_ctrl_time_frac_career_avg,diff_prior_title_fights,diff_static_age,diff_static_reach_cm,diff_static_ape_index_cm,same_stance
7839,2024-11-09,Welterweight Bout,Carlos Prates,Neil Magny,True,0.066144,4.0,1.025963,0.032354,0.002895,-0.080955,0.0,-6.0,-5.08,2.842171e-14,False
7834,2024-11-09,Women's Strawweight Bout,Gillian Robertson,Luana Pinheiro,True,0.063482,4.0,-2.566507,0.046631,0.023965,0.264371,0.0,-2.0,2.54,-5.080000e+00,True
7829,2024-11-09,Middleweight Bout,Dusko Todorovic,Mansur Abdul-Malik,False,NaN,-1.0,NaN,NaN,NaN,NaN,0.0,3.0,-12.70,-1.016000e+01,True
7830,2024-11-09,Middleweight Bout,Antonio Trocoli,Tresean Gore,False,-0.028758,-2.0,-0.373803,-0.013541,0.019997,0.046426,0.0,3.0,10.16,-5.080000e+00,False
7832,2024-11-09,Bantamweight Bout,Da'Mon Blackshear,Cody Stamann,True,-0.077902,0.0,-0.238998,0.001923,0.070975,0.026952,0.0,-5.0,20.32,1.016000e+01,False
7833,2024-11-09,Welterweight Bout,Charles Radtke,Matthew Semelsberger,True,0.071623,2.0,-1.221077,0.041519,0.005274,0.086817,0.0,3.0,-7.62,2.540000e+00,False
7831,2024-11-09,Bantamweight Bout,Cortavious Romious,Gaston Bolanos,False,NaN,1.0,NaN,NaN,NaN,NaN,0.0,-2.0,-2.54,5.080000e+00,False
7835,2024-11-09,Women's Bantamweight Bout,Klaudia Sygula,Melissa Mullins,False,NaN,1.0,NaN,NaN,NaN,NaN,0.0,-8.0,2.54,0.000000e+00,True
7836,2024-11-09,Middleweight Bout,Reinier de Ridder,Gerald Meerschaert,True,NaN,-2.0,NaN,NaN,NaN,NaN,0.0,-2.0,2.54,-5.080000e+00,True
7837,2024-11-09,Welterweight Bout,Zachary Scroggin,Elizeu Zaleski dos Santos,False,NaN,1.0,NaN,NaN,NaN,NaN,0.0,-9.0,2.54,-7.620000e+00,True


In [11]:
# load the saved wide table back in a fresh session, for inspection
from ufc_predictor.data.io import load_processed

df = load_processed("fights_featured")
df.shape

(7840, 367)

## The X/y manifest

366 columns are not interchangeable. 46 of them are the raw per-fight stats
of *the bout being predicted*, sitting one word away from their safe
counterparts — `fighter_a_sig_strikes` is the answer sheet,
`fighter_a_prior_sig_strikes_accuracy_career_avg` is a feature. So `X` gets
built from an explicit allow-list, never by dropping columns.

`manifest.py` groups the safe features by what they measure, so the choice of
what to train on can be made by measurement rather than up front. Two guards
back it: no raw post-fight column can be selected, and every column in the
table must be classified — add one without deciding what it is and the test
suite fails rather than letting it drift silently into or out of `X`.

In [12]:
from ufc_predictor.features import manifest

# completeness guard: nothing in the table is unaccounted for
unclassified = manifest.unclassified_columns(featured)
assert unclassified == [], f"unclassified columns: {unclassified}"

# make_xy refuses rows with no winner, so go through the split, which drops
# draws/no-contests and keeps only the 2014+ era
from ufc_predictor.modeling.split import chronological_split

train, validation, test = chronological_split(featured)
print(f"train {len(train):,}   validation {len(validation):,}   test {len(test):,}")

X, y = manifest.make_xy(train)
print(f"X: {X.shape[0]} rows x {X.shape[1]} features     y: {y.shape[0]} rows")
print(f"leaky columns held out of X: {len(manifest.leaky_columns(featured))}")
print(f"label in X? {manifest.LABEL_COLUMN in X.columns}")

train 2,862   validation 941   test 1,471


X: 2862 rows x 105 features     y: 2862 rows
leaky columns held out of X: 46
label in X? False


In [13]:
import pandas as pd
# What each feature group costs in missing rows.
#
# Measured on the modelling rows only (2014+, decided).
# Nothing here is imputed - the remaining NaNs are real.
report = manifest.nan_report(pd.concat([train, validation, test]))
report["rows_with_nan"] = (report["rows_with_nan"] * 100).round(1)
print(report.to_string(index=False))

print(
    "\nAfter shrinkage a missing value means one of two things: a debutant (no UFC"
    "\nhistory - genuinely unknown, left for the model to handle) or a gap in the raw"
    "\nfighter table (reach, stance, height). The 0/0 accuracy NaNs are gone: a fighter"
    "\nwho never attempted something now gets the league value."
)

     group  columns  rows_with_nan
cold_start        2            0.0
  physical        5            4.7
    record        8           21.4
      pace        2           21.4
  striking       32           21.4
 grappling       10           21.4
    damage        2           21.4
   defence       44           21.4
       ALL      105           23.2

After shrinkage a missing value means one of two things: a debutant (no UFC
history - genuinely unknown, left for the model to handle) or a gap in the raw
fighter table (reach, stance, height). The 0/0 accuracy NaNs are gone: a fighter
who never attempted something now gets the league value.
